# Day 32 - ONNX 模型部署

> 目标: 把训练好的模型导出为 ONNX, 跨平台部署
>
> ONNX = Open Neural Network Exchange — 模型的"通用语言"

---

## 为什么需要 ONNX?

```
训练阶段:                          部署阶段:
  PyTorch / TF -> 训练模型          需要: 跨平台、高性能推理
  依赖 GPU + 框架                   需要: CPU / 手机 / 边缘设备
                                                                 
ONNX 的作用:
  PyTorch ─┐                        ┌─ CPU (ONNX Runtime)
  TensorFlow ─┤    ONNX     ───►    ├─ GPU (TensorRT)
  Scikit-learn ─┤   (通用格式)       ├─ 手机 (CoreML/NNAPI)
               ┘                    └─ Web (ONNX.js / WebNN)
```

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt

print('=' * 40)
print('Day 32 - ONNX Deployment')
print('=' * 40)

Day 32 - ONNX Deployment


## 1. 训练一个简单模型

> 用 Day 15 的 RNN 做正弦波预测, 然后导出 ONNX

In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================
# 1. 数据准备 (保持不变)
# ============================================
seq_len = 10
total = 1000
t = np.linspace(0, 100 * np.pi, total, dtype=np.float32)
data = np.sin(t)

def make_seqs(data, seq_len):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len])
        ys.append(data[i+seq_len])
    return np.array(xs), np.array(ys)

xs, ys = make_seqs(data, seq_len)
split = int(0.8 * len(xs))
x_train = torch.FloatTensor(xs[:split]).unsqueeze(-1)  # (N, seq_len, 1)
y_train = torch.FloatTensor(ys[:split]).unsqueeze(-1)  # (N, 1)

# ============================================
# 2. NanoGPT 风格模型
# ============================================
class NanoGPT(nn.Module):
    def __init__(self, seq_len=10, d_model=64, n_heads=4, n_layers=3, dim_feedforward=128):
        super().__init__()
        self.seq_len = seq_len
        self.d_model = d_model
        
        # 输入投影: 1 -> d_model
        self.input_proj = nn.Linear(1, d_model)
        
        # 可学习位置编码 (固定长度，ONNX友好)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        
        # Transformer Decoder Layers (NanoGPT 核心)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=0.0,          # 导出ONNX时建议关闭dropout
            activation='gelu',    # GPT系列标配激活函数
            batch_first=True,     # ⚠️ 关键：避免transpose，ONNX更兼容
            norm_first=True,      # Pre-Norm架构，训练更稳定
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        
        # 输出头: d_model -> 1
        self.output_head = nn.Linear(d_model, 1)
        
        # 因果掩码 (注册为buffer，不参与梯度更新，但会随模型保存)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        self.register_buffer('causal_mask', causal_mask)

    def forward(self, x):
        # x: (batch, seq_len, 1)
        B, T, _ = x.shape
        
        # 输入投影 + 位置编码
        h = self.input_proj(x)  # (B, T, d_model)
        positions = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        h = h + self.pos_emb(positions)
        
        # Transformer解码 (带因果掩码)
        h = self.transformer(h, memory=h, tgt_mask=self.causal_mask[:T, :T])
        
        # 取最后一个时间步的输出做预测 (自回归风格)
        last_hidden = h[:, -1:, :]  # (B, 1, d_model) 保持3D避免squeeze
        out = self.output_head(last_hidden)  # (B, 1, 1)
        return out  # ONNX导出时保持3D，推理时再squeeze

# ============================================
# 3. 训练
# ============================================
model = NanoGPT(seq_len=seq_len)
opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)  # GPT标配优化器
criterion = nn.MSELoss()

for epoch in range(200):  # Transformer收敛较慢，增加epoch
    opt.zero_grad()
    pred = model(x_train)           # (N, 1, 1)
    loss = criterion(pred, y_train.unsqueeze(1))  # y_train也保持3D对齐
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪防爆炸
    opt.step()
    
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:>3}/200 | Loss: {loss.item():.6f}')

print(f'\nTraining done! Final loss: {loss.item():.6f}')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# ============================================
# 4. 导出 ONNX
# ============================================
dummy_input = torch.randn(1, seq_len, 1)  # 固定batch=1用于导出

torch.onnx.export(
    model,
    dummy_input,
    'nanogpt_sine.onnx',
    opset_version=17,               # ≥14 才完整支持 Transformer/GELU
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={                  # 允许batch维度动态
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    do_constant_folding=True,       # 常量折叠优化
)
print('\n✅ ONNX exported to nanogpt_sine.onnx')

# 验证导出是否成功
import onnxruntime as ort
session = ort.InferenceSession('nanogpt_sine.onnx')
test_input = x_train[:4].numpy()
onnx_pred = session.run(None, {'input': test_input})[0]
py_pred = model(x_train[:4]).detach().numpy()
print(f'ONNX vs PyTorch max diff: {np.abs(onnx_pred - py_pred).max():.2e}')

Epoch  50/200 | Loss: 0.036719
Epoch 100/200 | Loss: 0.005805
Epoch 150/200 | Loss: 0.000005


D:\hyy\Temp\ipykernel_20764\3328034925.py:103: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
D:\hyy\Temp\ipykernel_20764\3328034925.py:103: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0720 17:54:27.399000 20764 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Plea

Epoch 200/200 | Loss: 0.000001

Training done! Final loss: 0.000001
Model params: 151,553


W0720 17:54:27.917000 20764 site-packages\torch\fx\experimental\symbolic_shapes.py:8866] Unable to find user code corresponding to {u0}


[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=True)`...





def forward(self, arg0_1: "f32[64, 1]", arg1_1: "f32[64]", arg2_1: "f32[10, 64]", arg3_1: "f32[192, 64]", arg4_1: "f32[192]", arg5_1: "f32[64, 64]", arg6_1: "f32[64]", arg7_1: "f32[192, 64]", arg8_1: "f32[192]", arg9_1: "f32[64, 64]", arg10_1: "f32[64]", arg11_1: "f32[128, 64]", arg12_1: "f32[128]", arg13_1: "f32[64, 128]", arg14_1: "f32[64]", arg15_1: "f32[64]", arg16_1: "f32[64]", arg17_1: "f32[64]", arg18_1: "f32[64]", arg19_1: "f32[64]", arg20_1: "f32[64]", arg21_1: "f32[192, 64]", arg22_1: "f32[192]", arg23_1: "f32[64, 64]", arg24_1: "f32[64]", arg25_1: "f32[192, 64]", arg26_1: "f32[192]", arg27_1: "f32[64, 64]", arg28_1: "f32[64]", arg29_1: "f32[128, 64]", arg30_1: "f32[128]", arg31_1: "f32[64, 128]", arg32_1: "f32[64]", arg33_1: "f32[64]", arg34_1: "f32[64]", arg35_1: "f32[64]", arg36_1: "f32[64]", arg37_1: "f32[64]", arg38_1: "f32[64]", arg39_1: "f32[192, 64]", arg40_1: "f32[192]", arg41_1: "f32[64, 64]", arg42_1: "f32[64]", arg43_1: "f32[192, 64]", arg44_1: "f32[192]", arg4

[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'torch.fx.experimental.symbolic_shapes.GuardOnDataDependentSymNode'>: Could not guard on data-dependent expression Eq(u0, 1) (unhinted: Eq(u0, 1)).  (Size-like symbols: none)

consider using data-dependent friendly APIs such as guard_or_false, guard_or_true and statically_known_true.
Caused by: (_export\non_strict_utils.py:1205 in __torch_function__)
For more information, run with TORCH_LOGS="dynamic"
For extended logs when we create symbols, also add TORCH

## 2. 导出 ONNX

> PyTorch → ONNX: 一行代码搞定

In [6]:
model.eval()

# 导出一个 batch 的 dummy 输入
dummy = torch.randn(1, 1, 10)  # (batch=1, channels=1, seq_len=10)

torch.onnx.export(
    model,
    dummy,
    'model.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=17,
)

print('Exported: model.onnx')
print(f'ONNX file size: {os.path.getsize("model.onnx") / 1024:.1f} KB')

D:\hyy\Temp\ipykernel_20764\3450152668.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0720 17:54:34.139000 20764 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
E0720 17:54:34.632000 20764 site-packages\torch\_subclasses\fake_tensor.py:3090] failed while attempting to run meta for aten.mm.default
E0720 17:54:34.632000 20764 site-packages\torch\_subclasses\fake_

[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=True)`...


E0720 17:54:34.853000 20764 site-packages\torch\export\_trace.py:1323] always_classified is unsupported.
E0720 17:54:34.854000 20764 site-packages\torch\export\_trace.py:1323] always_classified is unsupported.


[torch.onnx] Obtain model graph for `NanoGPT([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'RuntimeError'>: a and b must have same reduction dim, but got [s77, 10] X [1, 64].

(Refer to the full stack trace above for more information.)

## 3. ONNX Runtime 推理

> 用 ONNX Runtime 加载并推理, 不依赖 PyTorch

In [7]:
try:
    import onnxruntime as ort

    session = ort.InferenceSession('model.onnx')
    input_name = session.get_inputs()[0].name

    test_input = np.random.randn(1, 1, 10).astype(np.float32)
    result = session.run(None, {input_name: test_input})[0]

    print(f'ONNX inference result shape: {result.shape}')
    print(f'Result: {result[0, 0]:.6f}')
    print('ONNX Runtime works without PyTorch!')

except ImportError:
    print('onnxruntime not installed. Use cell-9 instead.')

ONNX inference result shape: (1, 1)
Result: 0.045750
ONNX Runtime works without PyTorch!


## 4. 性能对比: PyTorch vs ONNX

> ONNX Runtime 通常比 PyTorch 原生推理快 (尤其是 CPU 上)

In [8]:
import onnxruntime as ort

session = ort.InferenceSession('model.onnx')
input_name = session.get_inputs()[0].name

batch_sizes = [1, 16, 64, 256]
trials = 500

pytorch_times = []
onnx_times = []

for bs in batch_sizes:
    x = torch.randn(bs, 1, 10)
    start = time.time()
    with torch.no_grad():
        for _ in range(trials):
            model(x)
    pt_time = (time.time() - start) / trials * 1000  # ms
    pytorch_times.append(pt_time)

    x_np = x.numpy()
    start = time.time()
    for _ in range(trials):
        session.run(None, {input_name: x_np})
    ort_time = (time.time() - start) / trials * 1000
    onnx_times.append(ort_time)

    faster = 'ONNX' if ort_time < pt_time else 'PyTorch'
    print(f'Batch={bs:>3d}  |  PyTorch: {pt_time:.3f}ms  |  ONNX: {ort_time:.3f}ms  |  {faster} faster')

plt.figure(figsize=(8, 4))
x = range(len(batch_sizes))
plt.bar([i - 0.15 for i in x], pytorch_times, width=0.3, label='PyTorch', color='#e17055')
plt.bar([i + 0.15 for i in x], onnx_times, width=0.3, label='ONNX Runtime', color='#00b894')
plt.xticks(x, [f'BS={b}' for b in batch_sizes])
plt.ylabel('Time (ms)'); plt.title('PyTorch vs ONNX Inference Speed')
plt.legend(); plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('day32_speed_comparison.png', dpi=100)
plt.show()

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x10 and 1x64)

## 5. 从 ONNX 到落地

### ONNX 的优势

```
1. 跨平台: Windows / Linux / Mac / 手机 / 浏览器
2. 跨框架: PyTorch / TF / JAX → ONNX → 任意框架
3. 优化: 算子融合 + 常量折叠 + 量化
4. 小: 去掉训练代码, 只有推理图
```

### 部署流程

```
训练 (PyTorch)
  → 导出 ONNX
  → 优化 (onnx-simplifier / onnx optimizer)
  → 量化 (INT8/FP16) → 更小更快
  → 部署:
       ├── ONNX Runtime (CPU/GPU, 标准方案)
       ├── TensorRT (NVIDIA GPU, 最快)
       ├── OpenVINO (Intel CPU, 边缘设备)
       ├── CoreML (Apple 设备)
       └── TFLite / NNAPI (Android)
```

### 量化效果

| 精度 | 模型大小 | 速度 | 精度损失 |
|:----|:--------:|:----:|:-------:|
| FP32 | 100% | 1x | 0 |
| FP16 | 50% | 1.5-2x | ~0% |
| INT8 | 25% | 2-4x | ~1-2% |

> 工业界: INT8 量化后部署到手机/边缘设备
> 精度掉 1% 但速度快 4 倍, 体积减少 75%

---

# Day 32 完成!

## 今天做了什么

```
1. 训练一个 PyTorch 模型
2. 导出为 ONNX (model.onnx, 几 KB)
3. 用 ONNX Runtime 加载并推理
4. 对比 PyTorch vs ONNX 推理速度
5. 了解量化 + 部署全流程
```

## 剩余内容

| Day | 内容 |
|:---:|:----|
| **33** | 整理 GitHub + 写技术博客 |
| **34** | 方向选择 (CV/NLP/多模态/RL) |
| **35** | 最终总结 + 知识地图 |

## 作业 (2 题)

### 1. 对比不同 batch size
**位置:** cell-9, `batch_sizes = [1, 16, 64, 256]`

ONNX 在大 batch 下优势更明显吗?

<details>
<summary>📖 答案</summary>
ONNX Runtime 对大 batch 有额外优化
通常 batch=1 时差距不大
batch=64+ 时 ONNX 可能快 2-3 倍
因为 ONNX Runtime 可以并行执行算子
</details>

### 2. 导出更复杂的模型
**位置:** cell-3, `SimpleModel`

换成 Day 24 的 NanoGPT, 导出 ONNX 试试

<details>
<summary>📖 答案</summary>
NanoGPT 有动态控制流 (for 循环), 导出 ONNX 会报错
解决方案: 用 `torch.jit.trace` + 固定输入长度
或者把生成循环剥离, 只导出单步推理
</details>